# Explorative Data Analysis (EDA) of the ANZ Synthesis Transactional Dataset

In [251]:
# Importing Required Libraries
import numpy as np
import pandas as pd 
import plotly.express as px
import warnings
import math
import plotly.graph_objects as go
import plotly.express as px
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings('ignore')

## 1. Data Importing

In [252]:
# 1. Importing the ANZ Synthesised Transaction Dataset
df_raw = pd.read_csv('anz_internship_raw_data.csv')
df_raw.head()   # Show the top 5 rows

,status,card_present_flag,bpay_biller_code,account,currency,long_lat,txn_description,merchant_id,merchant_code,first_name,...,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,country,customer_id,merchant_long_lat,movement
0,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,NaN,Diana,...,26,Ashmore,QLD,2018-08-01T01:01:15.000+0000,16.25,a623070bfead4541a6b0fff8a09e706c,Australia,CUS-2487424745,153.38 -27.99,debit
1,authorized,0.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,NaN,Diana,...,26,Sydney,NSW,2018-08-01T01:13:45.000+0000,14.19,13270a2a902145da9db4c951e04b51b9,Australia,CUS-2487424745,151.21 -33.87,debit
2,authorized,1.0,NaN,ACC-1222300524,AUD,151.23 -33.94,POS,835c231d-8cdf-4e96-859d-e9d571760cf0,NaN,Michael,...,38,Sydney,NSW,2018-08-01T01:26:15.000+0000,6.42,feb79e7ecd7048a5a36ec889d1a94270,Australia,CUS-2142601169,151.21 -33.87,debit
3,authorized,1.0,NaN,ACC-1037050564,AUD,153.10 -27.66,SALES-POS,48514682-c78a-4a88-b0da-2d6302e64673,NaN,Rhonda,...,40,Buderim,QLD,2018-08-01T01:38:45.000+0000,40.90,2698170da3704fd981b15e64a006079e,Australia,CUS-1614226872,153.05 -26.68,debit
4,authorized,1.0,NaN,ACC-1598451071,AUD,153.41 -27.95,SALES-POS,b4e02c10-0852-4273-b8fd-7b3395e32eb0,NaN,Diana,...,26,Mermaid Beach,QLD,2018-08-01T01:51:15.000+0000,3.25,329adf79878c4cf0aeb4188b4691c266,Australia,CUS-2487424745,153.44 -28.06,debit


In [253]:
df_raw.shape # The data contains 12043 rows and 23 columns

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   status             12043 non-null  object 
 1   card_present_flag  7717 non-null   float64
 2   bpay_biller_code   885 non-null    object 
 3   account            12043 non-null  object 
 4   currency           12043 non-null  object 
 5   long_lat           12043 non-null  object 
 6   txn_description    12043 non-null  object 
 7   merchant_id        7717 non-null   object 
 8   merchant_code      883 non-null    float64
 9   first_name         12043 non-null  object 
 10  balance            12043 non-null  float64
 11  date               12043 non-null  object 
 12  gender             12043 non-null  object 
 13  age                12043 non-null  int64  
 14  merchant_suburb    7717 non-null   object 
 15  merchant_state     7717 non-null   object 
 16  extraction         120

## 2. Data Cleaning

In [254]:
# 'bpay_biller_code' & 'merchant_code' have the majority of null values, so the rows of missing data must be dropped

df_raw.isnull().sum() # showing all rows that contain any columns with null values

df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   status             12043 non-null  object 
 1   card_present_flag  7717 non-null   float64
 2   bpay_biller_code   885 non-null    object 
 3   account            12043 non-null  object 
 4   currency           12043 non-null  object 
 5   long_lat           12043 non-null  object 
 6   txn_description    12043 non-null  object 
 7   merchant_id        7717 non-null   object 
 8   merchant_code      883 non-null    float64
 9   first_name         12043 non-null  object 
 10  balance            12043 non-null  float64
 11  date               12043 non-null  object 
 12  gender             12043 non-null  object 
 13  age                12043 non-null  int64  
 14  merchant_suburb    7717 non-null   object 
 15  merchant_state     7717 non-null   object 
 16  extraction         120

In [255]:
# Suspected redundant columns
list_names = ['bpay_biller_code', 'card_present_flag', 'merchant_code', 'merchant_suburb', 'merchant_state']

for element in list_names:
    print(f"{element}: {df_raw[element].unique()}")

# Summary
# The 'bpay_biller_code' and 'merchant_code' do not have consistent values - So they are to be removed
columns_to_remove = ['bpay_biller_code', 'merchant_code']

bpay_biller_code: [nan '0' ' THE DISCOUNT CHEMIST GROUP'
 ' LAND WATER & PLANNING East Melbourne']
card_present_flag: [ 1.  0. nan]
merchant_code: [nan  0.]
merchant_suburb: ['Ashmore' 'Sydney' 'Buderim' ... 'Donvale' 'Ulmarra' 'Kings Park']
merchant_state: ['QLD' 'NSW' nan 'VIC' 'WA' 'SA' 'NT' 'TAS' 'ACT']


In [256]:
# Checking for unique values of each column

df = df_raw.copy()
one_count_columns = []

for col in df_raw.columns:
    
    # drop column if the count is 1 or in columns_to_remove list
    if (len(df_raw[col].unique()) == 1) or (col in columns_to_remove):
        one_count_columns.append(col)

df.drop(one_count_columns, axis=1, inplace=True)

In [257]:
# Chicking if there are any duplicate records in the dataset
if df.duplicated().sum() == 0:
    print("There are no duplicates.")
    df_cleaned = df.copy()  # Copying the cleaned dataset
    
else:
    print("There are duplicates in the dataset.")

There are no duplicates.


## 3. Data Exploration and Analysis

In [258]:
# Separate the data by age to analyse data per age group
df_cleaned['age_group'] = pd.cut(df_cleaned['age'], [0, 20, 30, 40, 50, 60, max(df_cleaned['age'])], 
                            labels=['<20', '20-30', '30-40', '40-50', '50-60', '>60'])

df_cleaned.head(2)

,status,card_present_flag,account,long_lat,txn_description,merchant_id,first_name,balance,date,gender,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,customer_id,merchant_long_lat,movement,age_group
0,authorized,1.0,ACC-1598451071,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,Diana,35.39,8/1/2018,F,26,Ashmore,QLD,2018-08-01T01:01:15.000+0000,16.25,a623070bfead4541a6b0fff8a09e706c,CUS-2487424745,153.38 -27.99,debit,20-30
1,authorized,0.0,ACC-1598451071,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,Diana,21.20,8/1/2018,F,26,Sydney,NSW,2018-08-01T01:13:45.000+0000,14.19,13270a2a902145da9db4c951e04b51b9,CUS-2487424745,151.21 -33.87,debit,20-30


In [259]:
# Convert the datatype of 'date' and 'extraction' columns to datetime datatype
df_cleaned.loc[:, ['date', 'extraction']] = df_cleaned.loc[:, ['date', 'extraction']].apply(pd.to_datetime, errors='coerce')

df_cleaned.head(2) # Confirm if the datatype of the 'date' and 'extraction' columns have been updated

,status,card_present_flag,account,long_lat,txn_description,merchant_id,first_name,balance,date,gender,age,merchant_suburb,merchant_state,extraction,amount,transaction_id,customer_id,merchant_long_lat,movement,age_group
0,authorized,1.0,ACC-1598451071,153.41 -27.95,POS,81c48296-73be-44a7-befa-d053f48ce7cd,Diana,35.39,2018-08-01 00:00:00,F,26,Ashmore,QLD,2018-08-01 01:01:15+00:00,16.25,a623070bfead4541a6b0fff8a09e706c,CUS-2487424745,153.38 -27.99,debit,20-30
1,authorized,0.0,ACC-1598451071,153.41 -27.95,SALES-POS,830a451c-316e-4a6a-bf25-e37caedca49e,Diana,21.20,2018-08-01 00:00:00,F,26,Sydney,NSW,2018-08-01 01:13:45+00:00,14.19,13270a2a902145da9db4c951e04b51b9,CUS-2487424745,151.21 -33.87,debit,20-30


In [260]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   status             12043 non-null  object  
 1   card_present_flag  7717 non-null   float64 
 2   account            12043 non-null  object  
 3   long_lat           12043 non-null  object  
 4   txn_description    12043 non-null  object  
 5   merchant_id        7717 non-null   object  
 6   first_name         12043 non-null  object  
 7   balance            12043 non-null  float64 
 8   date               12043 non-null  object  
 9   gender             12043 non-null  object  
 10  age                12043 non-null  int64   
 11  merchant_suburb    7717 non-null   object  
 12  merchant_state     7717 non-null   object  
 13  extraction         12043 non-null  object  
 14  amount             12043 non-null  float64 
 15  transaction_id     12043 non-null  object  
 16  cust

In [261]:
# Summary: The 'date' and 'extraction' are object datatype, they must be datetime
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'], errors ='coerce')
df_cleaned['extraction'] = pd.to_datetime(df_cleaned['extraction'], errors ='coerce')

# Create date helper columns
df_cleaned['month'] = df_cleaned['date'].dt.month_name()
df_cleaned['day'] = df_cleaned['date'].dt.day_name()
df_cleaned['hour']= df_cleaned['extraction'].dt.hour
df_cleaned.head()

# Change datatype of card_present_flag to INT
df.card_present_flag = df.card_present_flag.astype('Int64')
df.head()

df_cleaned.info() # Check datatypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   status             12043 non-null  object             
 1   card_present_flag  7717 non-null   float64            
 2   account            12043 non-null  object             
 3   long_lat           12043 non-null  object             
 4   txn_description    12043 non-null  object             
 5   merchant_id        7717 non-null   object             
 6   first_name         12043 non-null  object             
 7   balance            12043 non-null  float64            
 8   date               12043 non-null  datetime64[ns]     
 9   gender             12043 non-null  object             
 10  age                12043 non-null  int64              
 11  merchant_suburb    7717 non-null   object             
 12  merchant_state     7717 non-null   object     

In [262]:
# Key Observation

# The Non-Null Count is not the same for all columns
df_cleaned.isnull().sum() # showing all rows that contain any columns with null values

status                  0
card_present_flag    4326
account                 0
long_lat                0
txn_description         0
merchant_id          4326
first_name              0
balance                 0
date                    0
gender                  0
age                     0
merchant_suburb      4326
merchant_state       4326
extraction              0
amount                  0
transaction_id          0
customer_id             0
merchant_long_lat    4326
movement                0
age_group               0
month                   0
day                     0
hour                    0
dtype: int64

In [263]:
# Check the number of columns with null values
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12043 entries, 0 to 12042
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype              
---  ------             --------------  -----              
 0   status             12043 non-null  object             
 1   card_present_flag  7717 non-null   float64            
 2   account            12043 non-null  object             
 3   long_lat           12043 non-null  object             
 4   txn_description    12043 non-null  object             
 5   merchant_id        7717 non-null   object             
 6   first_name         12043 non-null  object             
 7   balance            12043 non-null  float64            
 8   date               12043 non-null  datetime64[ns]     
 9   gender             12043 non-null  object             
 10  age                12043 non-null  int64              
 11  merchant_suburb    7717 non-null   object             
 12  merchant_state     7717 non-null   object     

## 4. Analysis of Categorical variables

In [264]:
# Metadata of the dataset

# status - denotes the status of the transaction
# card_present_flag - did the customer use card for the transaction. 1 for yes and 0 for no
# bpay_biller_code - unique code of the BPay Transaction done by the customer
# account - customer's account number
# currency - Australian Dollar
# long_lat - transaction's location
# txn_description - mode of transaction. POS or SALES-POS
# merchant_id - merchant's id
# merchant_code - unique merchant code for each customers
# first_name - customer's first name

In [265]:
# Analysis of the dataset
# 4.1   Total available records in the dataset
# 4.2   Top 10 customers by transaction amount
# 4.3   Analysis of categorical variables (frequency / percentage)
# 4.4   Analysis of the mode of transaction (SALES-POS vs POS)
# 4.5   Analysis the total amount transacted by surburb
# 4.6   Analyzing the debit transactions
# 4.7   Transacted amount by state
# 4.8   Daily transacted amount
# 4.9   Transacted amount per day by gender
# 4.10  Total number of transactions by month
# 4.11  Analysis of the total amount transacted by month
# 4.12  Analysis of the number of transactions by age group
# 4.13  Transacted amount by age group and gender
# 4.14  Average transaction amount and transaction count by date
# 4.15  Average transaction amount by hour of the day
# 4.16  Number of Transactions by Hour of the Day, Month, and Gender
# 4.17  Analysing Credit Transactions - Top Valuable Customers by Average Balance
# 4.18  Average transaction amount by month

# 4.1   Total available records in the dataset

In [266]:
number_of_months = df_cleaned['month'].nunique() # Total number of months in the dataset
total_number_of_records = df_cleaned['transaction_id'].nunique() # Total available records in the dataset

# The start and end date of the dataset
start_date = df_cleaned['date'].min()
end_date = df_cleaned['date'].max()
data_duration = end_date - start_date

print(f"Total available day's worth of data: {data_duration}")
print(f"Total available records in the dataset: {total_number_of_records}")
print(f"Total number of months in the dataset: {number_of_months}")

Total available day's worth of data: 91 days 00:00:00
Total available records in the dataset: 12043
Total number of months in the dataset: 3


# 4.2   Top 10 customers by transaction amount

In [267]:
df_cleaned['customer_id'].nunique() # 1000 unique customers

# Top 10 customers by transaction amount
df_top_10_customers = df_cleaned.groupby(by='customer_id')[['amount']].sum().sort_values(by='amount', ascending=False).head(10).reset_index()

# Plot the top 10 customers by transaction amount
fig = px.bar(data_frame=df_top_10_customers, x='customer_id', y='amount', text=df_top_10_customers.amount.apply(lambda x : str(round(x/1000,2))+'k'), color='customer_id')
fig.update_traces(textposition='outside')
fig.update_xaxes(title='Customer ID')
fig.update_yaxes(title='Transaction Amount (Australian Dollar)')
fig.update_layout(title=dict(text = "Top 10 Customers by Transaction Amount (Australian Dollar)", x = 0.5, y = 0.95),
                  title_font_size=20, showlegend=False, height = 500,)

fig.show()

df_top_10_customers['amount'].sum() # Total transaction amount of top 10 customers in Australian Dollar
df_cleaned['amount'].sum() # Total transaction amount of all customers in Australian Dollar

print("Total transaction amount of top 10 customers: ${:,.2f}".format(df_top_10_customers['amount'].sum()))
print("Total transaction amount of all customers: ${:,.2f}".format(df_cleaned['amount'].sum()))
print("Percentage of total transaction amount by top 10 customers: {:.2f}%".format((df_top_10_customers['amount'].sum() / df_cleaned['amount'].sum()) * 100))

Total transaction amount of top 10 customers: $384,697.88
Total transaction amount of all customers: $2,263,284.20
Percentage of total transaction amount by top 10 customers: 17.00%


# 4.3   Analysis of Categorical Variables (Frequency / Percentage)

In [268]:

# Configuration and Data Setup
columns_of_interest = ['status', 'card_present_flag', 'txn_description' , 'movement' , 'gender', 'merchant_state']
column_names = ['Transaction Status', 'Card Present Flag', 'Mode of Transaction' , 'Movement', 'Gender', 'Merchant State']

plot_title = "Analysis of Categorical Variables (Frequency / Percentage)"

# Subplot initialization - Number of rows and columns for the sublots
cols = 2
rows = math.ceil(len(columns_of_interest) / cols)

# Cleaned-up subplot titles
subplot_titles = column_names

fig = make_subplots(
    rows=rows, 
    cols=cols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.15, 
    vertical_spacing=0.12, shared_yaxes=True)

# Populate Subplots Dynamically
for index, col_name in enumerate(columns_of_interest):
    # Calculate row and column positions
    row = (index // cols) + 1
    col = (index % cols) + 1
    
    # Compute counts once to optimize performance
    counts = df_cleaned[col_name].value_counts()
    percentages = df_cleaned[col_name].value_counts(normalize=True) * 100
    
    # Generate text labels (e.g., "42 (15%)")
    text_labels = [f"{count}<br>({pct:.1f}%)" for count, pct in zip(counts, percentages)]
    
    fig.add_trace(go.Bar(x=counts.index, y=counts, name=col_name,
            text=text_labels, textposition='auto',
            hovertemplate="<b>Category:</b> %{x}<br><b>Count:</b> %{y}<extra></extra>"), 
            row=row, col=col)

# Global Layout and Styling Customization
fig.update_layout(title=dict(text=plot_title, 
        x=0.5, y=0.98, 
        xanchor='center', yanchor='top'), 
        title_font_size=22, 
        showlegend=False, 
        height=300 * rows,  # Scales perfectly with the number of rows
        margin=dict(l=50, r=50, t=100, b=50),
        template="plotly_white",  # Clean, modern aesthetic
        )

# Links all y-axes to scale together when you zoom or pan
fig.update_yaxes(title_text="Number of transactions")

fig.show()

In [269]:
# Key Observations:

# About 64.1% of transactions were authorized and the 35.9% were posted.
# The majority (64.1%) of the transactions were done using "SALES-POS" & "POS"
# Males tend to do more transactions as compared to females, with 8.4% more males than females
# More than 80% of transactions were done using a physical card from a customer
# 92.7% transactions were conducted using a debit and the rest using a credit.
# NSW, VIC, QLD are the most busy merchant states, where as ACT and TAS are the least busy.

# 4.4   Analysis of the mode of transaction (SALES-POS vs POS)

In [270]:
# Analysis the mode of transactions

df_grp_transaction_modes = df_cleaned.groupby(by='txn_description')['amount'].sum().round().reset_index()

# Generate the treemap
fig = px.treemap(df_grp_transaction_modes, path=['txn_description'], values='amount',
    color='amount', color_continuous_scale='Viridis'
)

# Polish the layout
fig.update_layout(
    title=dict(text="<b>Total Amount in Australian Dollar by Transaction Description</b>",
        x = 0.5,y = 0.95, xanchor = 'center'),
    margin=dict(l=10, r=10, t=70, b=10),
    coloraxis_showscale=False)

# Configure text display inside the boxes
fig.data[0].textinfo = 'label+value'

fig.show()

In [271]:
# Key Observations:

# Pay/Salary is the major contributor of bank transactional amounts. 
# This is expected as salary transaction amount is usually very high as compared to normal debit transactions.

# 4.5   Analysis the total amount transacted by surburb

In [272]:
# Analysis the total amount transacted by surburb
df_grp_amount_by_surburb = df_cleaned.groupby(by='merchant_suburb')['amount'].sum().round().reset_index()

# Generate the treemap
fig = px.treemap(df_grp_amount_by_surburb, path=['merchant_suburb'],
           values='amount', color = 'amount',)

# Polish the layout
fig.update_layout(title=dict(text = "Total Transactional Amount by Suburb", x = 0.5, y = 0.95),
                    margin=dict(l=10, r=10, t=50, b=10), showlegend=False,)

# Configure text display inside the boxes
fig.data[0].textinfo = 'label+value'
fig.update_traces(marker_coloraxis = None)
fig.show()

In [273]:
# Key Observation:

# Sydney, Melbourne, South Brisbane, and Mascot are the leading surburbs by the total value of transaction amount.

# 4.6   Analyzing the debit transactions

In [274]:
# Analyzing the debit transactions
df_debit_transactions = df_cleaned[df_cleaned['movement'] == 'debit'] # Debit Transactions
df_debit_transactions.shape # 11160 debit transactions

# Configuration and Data Setup
column_names = ['status', 'card_present_flag', 'txn_description', 'movement', 'gender', 'merchant_state']
plot_title = "Total Transactional Amount in Australian Dollar and Percentage"


# Dynamic grid generation
coloumns_per_row = 2
rows = math.ceil(len(column_names) / coloumns_per_row)

# Clean, professional subplot titles
subplot_titles = ['Transaction Status', 'Card Present Flag', 'Mode of Transaction' , 'Movement', 'Gender', 'Merchant State']

fig = make_subplots(
    rows=rows, 
    cols=coloumns_per_row,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.15, 
    vertical_spacing=0.15
)

# 2. Populate Subplots Dynamically
for index, col_name in enumerate(column_names):
    # Calculate exact subplot positioning
    row = (index // coloumns_per_row) + 1
    col = (index % coloumns_per_row) + 1
    
    # CRITICAL FIX: Group, aggregate, and sum ONCE per column to maximize performance
    grouped = df_debit_transactions.groupby(by=col_name)['amount'].sum().reset_index()
    
    # Calculate percentages accurately using the aggregated total
    total_amount = grouped['amount'].sum()
    
    # Create descriptive text labels (e.g., "$1,250\n(15.4%)")
    text_labels = [
        f"${amt:,.0f}<br>({(amt / total_amount) * 100:.1f}%)" 
        if total_amount > 0 else "$0<br>(0%)"
        for amt in grouped['amount']
    ]
    
    fig.add_trace(
        go.Bar(
            x=grouped[col_name],                      # FIX: Passed the category series
            y=grouped['amount'].round(2),             # FIX: Passed the numeric series
            name=col_name,
            text=text_labels,
            textposition='auto',
            hovertemplate="<b>Category:</b> %{x}<br><b>Total Amount:</b> $%{y:,.2f}<extra></extra>"
        ),
        row=row, 
        col=col
    )

fig.update_layout(
    title=dict(
        text=plot_title, 
        x=0.5, 
        y=0.98, 
        xanchor='center', 
        yanchor='top'
    ),
    title_font_size=22, 
    showlegend=False, 
    height=320 * rows,  # Scales perfectly regardless of how many columns you analyze
    margin=dict(l=60, r=60, t=120, b=60),
    template="plotly_white"  # Clean, modern layout look
)

# Global Layout and Styling Customization
fig.update_layout(title=dict(text=plot_title, 
        x=0.5, y=0.98, 
        xanchor='center', yanchor='top'), 
        title_font_size=22, 
        showlegend=False, 
        height=300 * rows,  # Scales perfectly with the number of rows
        margin=dict(l=50, r=50, t=100, b=50),
        template="plotly_white",  # Clean, modern aesthetic
        )

# Links all y-axes to scale together when you zoom or pan
fig.update_yaxes(title_text="Value of transactions")

fig.show()

In [275]:
# Key Obervations:

# 80% of the amount was transacted using cards.
# Payment mode of transaction contributes most to the transaction amount.
# NSW & VIC merchant states contributed more than half to overall transaction amount

# 4.7   Transactional Amount by State

In [276]:
df_grp_gender_by_state = df_cleaned.groupby(by=['merchant_state', 'gender'])[['amount']].sum().reset_index()
day_order = df_cleaned.groupby(by=['merchant_state'])[['amount']].sum().sort_values(by='amount',ascending=False).index
df_grp_gender_by_state['merchant_state'] = pd.Categorical(df_grp_gender_by_state['merchant_state'],day_order)
df_grp_gender_by_state = df_grp_gender_by_state.groupby(by=['merchant_state', 'gender']).sum().reset_index()

fig = px.bar(data_frame=df_grp_gender_by_state,
       x='merchant_state', y='amount', color='gender',
       barmode='group', text=df_grp_gender_by_state.amount.apply(lambda x : str(round(x/1000,2))+'k'))

fig.update_traces(textposition='outside')
fig.update_xaxes(title='Merchant State') 
fig.update_yaxes(title='Transaction Amount (Australian Dollar)')
fig.update_layout(title=dict(text = "Transaction Amount (Australian Dollar) in Merchant State by Gender", 
                             x=0.5, y=0.95),
                    title_font_size=20,)
fig.show()

In [277]:
# Key Observation:
# NSW and VIC are the primary drivers of transaction volume by a massive margin. 
# Combined, they account for roughly $189k in transactions, which eclipses all other states and territories combined.
# ACT and TAS represent negligible market share, with total transaction volumes lingering below $5k per state.

# In High-Volume Markets (NSW, VIC), Male spending strongly dominates. 
# For instance, in NSW, male transactions ($60.59k) outperform female transactions ($41.43k) by nearly 46%.
# In Mid-to-Low Volume Markets (QLD, WA, SA, NT), Female spending consistently outperforms male spending. 
# In states like Western Australia (WA) and South Australia (SA), women spend roughly double what men do (e.g., $11.35k vs $5.43k in SA).

# The Key Takeaway is that male-targeted campaigns should be primarily around NSW and VIC.
# Focus female-targeted campaigns or localized promotions in the expanding mid-tier states (QLD, WA, SA), where women are clearly driving the local transaction volume.

# 4.8   Daily Transactional Amount

In [278]:
# Weekdays in chronological order
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Convert the 'day' column to a Categorical type with this order
df_cleaned['day'] = pd.Categorical(df_cleaned['day'], categories=days_order, ordered=True)

# Compute value counts and sort by the days
day_counts = df_cleaned['day'].value_counts().sort_index()


# Pass cleanly to the Plotly chart variables
x = day_counts.index.tolist()
y = day_counts.tolist()



# For number of daily transactions
fig_number_of_transactions = px.bar(data_frame=df_cleaned, x = x, y = y,
       text=day_counts.tolist(), color=x)

fig_number_of_transactions.update_traces(textposition='outside', marker_coloraxis=None)
fig_number_of_transactions.update_xaxes(title='Day of the week') 
fig_number_of_transactions.update_yaxes(title='Number of Transactions')
fig_number_of_transactions.update_layout(title=dict(text = "Number of Transactions per Weekday", x=0.5, y=0.95),
                  title_font_size=20, showlegend=False, height = 450,)
fig_number_of_transactions.show()

# For the dollar value of the daily transactions
fig_value_of_transactions_per_day = px.bar(data_frame=df_cleaned.groupby(by='day')[['amount']].sum(),
            text=df_cleaned.groupby(by='day')[['amount']].sum()['amount'].apply(lambda x : str(round(x/1000,2))+'k'), color=day_counts.index.tolist())

fig_value_of_transactions_per_day.update_traces(textposition='outside')
fig_value_of_transactions_per_day.update_xaxes(title='Day of the week')
fig_value_of_transactions_per_day.update_yaxes(title='Transaction Amount (Australian Dollar)')
fig_value_of_transactions_per_day.update_layout(title=dict(text = "Transaction Amount (Australian Dollar) per Weekday", x=0.5, y=0.95),
                    title_font_size=20, showlegend=False, height = 450,)
fig_value_of_transactions_per_day.show()

In [279]:
# Observation:

# Observation:
# Transaction volume follows a clear mid-week peak pattern. Monday and Tuesday record the lowest activity, rising sharply to a Wednesday peak (2,063)
# before dipping on Thursday (1,801) and peaking again on Friday (2,073) - the highest day overall. 
# Weekend volumes (Saturday: 1,709; Sunday: 1,550) remain moderate, declining progressively into the new week. 
# This suggests transaction demand is strongly driven bb mid-week and end-of-work-week behaviour.

# Transaction Count Observation:
# Transaction value tells a strikingly different story to transaction count. While Wednesday
# and Friday led on volume, Monday emerges here as the second-highest value day (AUD 507.58k),
# trailing only Friday (AUD 516.91k) - suggesting Monday transactions are fewer but significantly higher in value. 
# Mid-week days (Tuesday: 329.48k, Wednesday: 402.73k,
# Thursday: 331.4k) cluster in a moderate range despite Wednesday's count dominance,
# implying that mid-week activity is driven by a high number of smaller-value transactions.
# The sharpest contrast is the weekend collapse: Saturday (93.0k) and Sunday (82.17k) drop
# to roughly 16% of Friday's value, even though their transaction counts were moderate -
# confirming that weekend transactions are both fewer and materially smaller in size.

# 4.9   Transaction Amount per day by Gender

In [280]:
# Transaction Amount per day by Gender

df_gender_grouping = df_cleaned.groupby(by=['day','gender'])[['amount']].sum().reset_index()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df_gender_grouping['day'] = pd.Categorical(df_gender_grouping['day'], day_order) 

df_gender_grouping = df_gender_grouping.groupby(by=['day','gender']).sum().reset_index()
fig = px.bar(data_frame=df_gender_grouping, x='day', y='amount', color='gender', barmode='group',
       text=df_gender_grouping.amount.apply(lambda x : str(round(x/1000,2))+'k'))

fig.update_traces(textposition='outside')
fig.update_xaxes(title='Weekday') 
fig.update_yaxes(title='Transaction Amount in Australian Dollar')
fig.update_layout(title=dict(text = "Transaction Amount (Australian Dollar) per day by Gender", x = 0.5, y = 0.95),
                  title_font_size=20,)
fig.show()

In [281]:
# Key Observations:

# Weekday spending (Mon–Fri) significantly dominates weekend spending across both genders, 
# suggesting the dataset largely reflects business-hour or salary-linked transactions 
# (e.g. recurring payments, bills, online purchases) rather than leisure spending.

# GENDER DIFFERENCES
# Males consistently outspend females on Monday and Friday, the two highest-volume days in the dataset,
# with peaks of AUD 304.98k (Mon) and AUD 303.53k (Fri).
# This 33.6% and 29.7% gender spedning difference on those days is the most pronounced in the chart
# and warrants further investigation (e.g. whether it correlates with higher male average salaries in the dataset).

# Females lead spending on Wednesday (AUD 207.87k vs 194.86k) and are competitive
# on Tuesday and Thursday. The female spend profile is more evenly distributed
# across the working week, while male spend is more front- and end-loaded.

# WEEKEND DROP-OFF
# Both genders show a sharp decline on Saturday and Sunday of roughly 80-85% lower than peak weekday volumes. 
# This is a meaningful structural pattern and likely reflects the transaction type mix 
# (e.g. fewer automated payments on weekends) rather than purely discretionary behaviour.

# NOTE ON OUTLIERS
# The Monday and Friday male peaks are notably higher than all other bars.
# These should be examined for outlier transactions that may be inflating the daily totals,
# a single large transfer on one day could distort the weekly average.

# 4.10  Total Number of Transactions per Month

In [282]:
# Total Number of Transactions per Month

# Month order for chronological sorting
month_order = ['August', 'September', 'October']

# Count transactions and reorder
month_counts = (df_cleaned['month'].value_counts().reindex(month_order, fill_value=0))

fig = px.bar(x=month_counts.index, y=month_counts.values,
    color=month_counts.values,
    text=month_counts.values)

fig.update_traces(textposition='outside')
fig.update_xaxes(title='Month')
fig.update_yaxes(title='Number of Transactions')
fig.update_layout(title=dict(text="Number of Transactions per Month",
        x=0.5, y=0.95), title_font_size=20, height=450)

fig.show()

In [283]:
# Key Observation: 

# There is a steady increase in the number of transactions by each passing month from October to November

# 4.11  Analysis of the total amount transacted by month

In [284]:
transaction_amount_by_month = df_cleaned.groupby(by='month')[['amount']].sum().reset_index()

fig = px.bar(transaction_amount_by_month, x='month', y='amount', text=transaction_amount_by_month['amount'].apply(lambda x : str(round(x/1000,2))+'k'),
       color=df_cleaned['month'].value_counts().tolist(),)

fig.update_traces(textposition='outside', marker_coloraxis=None)
fig.update_xaxes(title='Month')
fig.update_yaxes(title='Total Value of Transactions in Australian Dollar')
fig.update_layout(title=dict(text = "Total Value of Transactions in Australian Dollar by Month", x=0.5, y=0.95),
                    title_font_size=20, showlegend=False, height = 450,)
fig.show()

# 4.12  Analysis of the number of transactions by age group

In [285]:
# Number of Transactions by Age Group
number_of_transactions_by_age = df_cleaned['age_group'].value_counts().reindex(['<20', '20-30', '30-40', '40-50', '50-60', '>60'])


fig = px.bar(number_of_transactions_by_age,
       color=df_cleaned['age_group'].value_counts(),)

fig.update_traces(textposition='outside', marker_coloraxis=None)
fig.update_xaxes(title='Age Group')
fig.update_yaxes(title='Number of Transactions')
fig.update_layout(title=dict(text = "<b>Number of Transactions by Age Group</b>", x=0.5, y=0.95),
                    title_font_size=20, showlegend=False, height = 450,)
fig.show()

In [286]:
# Key Observations:

# DISTRIBUTION SHAPE
# Transaction counts follow a right-skewed distribution across age groups,
# peaking sharply in the 20-30 bracket (3,405 transactions) before declining steadily with age. 
# The 20-40 cohort collectively accounts for roughly 43% of all transactions in the datas, making them the dominant customer segment by activity volume.

# OLDER AGE GROUPS (50-60, >60)
# The sharp drop-off in the 50+ cohorts (224 and 150 transactions respectively)
# reflects low representation in the dataset rather than necessarily low banking
# activity in that demographic. Before making any business inference, it is worth
# checking whether this mirrors ANZ's actual customer age distribution or whether
# the synthesised dataset is skewed toward younger customers by design.

# WHAT THIS DOES NOT TELL US
# Transaction count alone does not indicate customer value. Older customers may
# transact less frequently but with higher individual amounts — a point worth
# cross-referencing against the salary prediction task in Task 2, where age
# may emerge as a meaningful predictor of income.

# 4.13  Transacted amount by age group and gender

In [287]:
df_gender = (df_cleaned.groupby(['age_group', 'gender'], as_index=False)['amount'].sum())
df_total = (df_cleaned.groupby('age_group', as_index=False)['amount'].sum())

fig = px.bar(df_gender, x='age_group', y='amount', color='gender', barmode='group',
    text=df_gender['amount'].apply(lambda x: f'{x/1000:.2f}k'))

fig.add_trace(go.Scatter(x=df_total['age_group'], y=df_total['amount'], mode='lines+markers',
        name='Total', line=dict(color='black', width=3)))

fig.update_traces(textposition='outside', selector=dict(type='bar'))

fig.update_layout(title=dict(text = "<b>Transacted Amount (Australian Dollar) by Age Group and Gender</b>", x=0.5, y=0.95),
                  xaxis_title="Age Group", 
                  yaxis_title="Transaction Amount (AUD)",
                  barmode='group', title_font_size=20, height = 450, showlegend=True,)

fig.show()

In [288]:
# Key Observation

# Total transaction amounts mirror the count distribution from the previous chart,
# peaking in the 20-30 and 30-40 brackets and declining sharply thereafter.
# However, cross-referencing the two charts reveals that the 20-30 group has 3,405 transactions 
# but generates AUD 839.39k in total spend, while the <20 group has a far higher transaction count
# (5,071) but only AUD 342.12k in total spend — implying the <20 group transacts
# far more frequently but in much smaller amounts per transaction.
# This reinforces the earlier suspicion that the <20 transaction count label
# may reflect a different metric, or that this cohort is characterised by
# high-frequency, low-value transactions (e.g. small daily purchases).

# MALE DOMINANCE IN PEAK EARNING YEARS
# Males outspend females in the 20-30 bracket by AUD 164.29k (+49%) and in the
# 30-40 bracket by AUD 94.92k (+31%). These are the prime working-age cohorts
# where salary differentials would be most pronounced — a pattern likely to
# surface again in Task 2 when salary is modelled against customer attributes.
# The gender spend gap appears to track the well-documented gender pay gap
# rather than purely behavioural differences.

# FEMALE LEAD IN <20 GROUP
# Females in the <20 bracket spend AUD 179.02k vs AUD 163.1k for males (+9.8%).
# This reversal is notable but should be treated cautiously — the absolute
# difference is modest (AUD 15.92k) and the <20 group has data quality concerns
# flagged in the previous chart. It may reflect a small number of high-value
# female transactions skewing the total rather than a structural pattern.

# OLDER COHORTS (50-60, >60)
# Males outspend females in both older brackets — AUD 32.24k vs 16.27k (50-60)
# and AUD 29.53k vs 10.59k (>60). The male-to-female ratio widens significantly
# here (~2x and ~2.8x respectively). With so few transactions in these groups
# (224 and 150 from the previous chart), these totals are highly sensitive to
# individual large transactions and should not be over-interpreted.

# ANALYTICAL LINK TO TASK 2
# The consistent male spend premium across 20-50 age groups, combined with the
# transaction count data, suggests that gender and age will likely be significant
# predictors in the salary regression model. However, causality should not be
# assumed — higher spend may reflect higher income, but it may also reflect
# different spending behaviour, debt levels, or household size.

# 4.14  Average transaction amount and transaction count by date

In [289]:
# Average transaction amount and transaction count by date
df_average_trans_amount_and_count = (df_cleaned.groupby('date').agg(
        Amount=('amount', 'mean'), Transaction_Count=('transaction_id', 'count')).reset_index())

fig = px.line(df_average_trans_amount_and_count, x='date', y=['Amount', 'Transaction_Count'], markers=True)

fig.update_layout(
    title=dict(text="<b>Average Transaction Amount vs Transaction Count Over Time</b>", x=0.5, y=0.95), 
    xaxis_title="Date", yaxis_title="Value",
    title_font_size=20, height=500, legend_title_text="")

fig.show()

In [290]:
# Observation:

# The average transaction amount on 7th August & Oct 21st was very high approx 100 AUD.
# Large number of transactions took place on 17th August & 28th September.

# 4.15  Average transaction amount by hour of the day

In [291]:
fig = px.line(df_cleaned.groupby(by='hour')[['amount']].sum(),
            text=df_cleaned.groupby(by='hour')['amount'].sum().apply(lambda x : str(round(x/1000))+'k').values)

fig.update_traces(line=dict(color="#F534E8", width=5))
fig.update_xaxes(title='Hour') 
fig.update_yaxes(title='Transaction Amount in Australian Dollar')
fig.update_layout(title=dict(text = "<b>Total Average Hourly Transacted Amount Per Day</b>", x=0.5, y=0.95),
                    title_font_size=20, showlegend=False, height = 450,)

fig.update_traces(textposition='middle right', fillcolor='red')
fig.show()

In [292]:
# Insights:

# Total transaction amount generated at 9:00 AM - 9:59 AM is approx 47k which is highest throughout the day.
# Between 12:00 AM - 7:00 AM we have least transaction amount because of off hours.

# 4.16  Number of Transactions by Hour of the Day, Month, and Gender

In [298]:
df_number_of_daily_transactions = (df_cleaned.groupby(['hour', 'month', 'gender'])
    .agg(Transaction_Count=('amount', 'count'), Total_Txn_Amount=('amount', 'sum')).reset_index())

fig = px.line(df_number_of_daily_transactions, x='hour', y='Transaction_Count',
    color='gender', facet_col='month', facet_col_wrap=3, markers=True,)
fig.update_traces(line=dict(width=4))
fig.update_xaxes(title='Hour of the Day')
fig.update_yaxes(title='Number of Transactions')
fig.update_layout(title=dict(text="<b>Number of Transactions by Hour of the Day, Month, and Gender</b>", x=0.5, y=0.96), 
                  title_font_size=20, height=450, legend_title_text="")

fig.show()

In [294]:
# Insights:

# In the month of September & October even though transaction count by females are more at 9:00 AM but TXN amount is still less. Seems like comparatively small transactions done by females during the start of the day.

# In October at 2:00 PM transaction amount by females is almost double as compared to males.

# 4.17  Analysing Credit Transactions

In [295]:
# Analysing Credit Transactions
df_top_10_clients_by_credit = df_cleaned[df_cleaned['movement']=='credit']
df_top_10_clients_by_credit.shape # 883 Credit transactions

fig=px.bar(df_top_10_clients_by_credit.groupby(by='customer_id')['balance'].mean().sort_values(ascending=False).head(10),
            text = df_top_10_clients_by_credit.groupby(by='customer_id')['balance'].mean().sort_values(ascending=False).head(10).apply(
                lambda x : str(round(x/1000,2))+'k'),
            color = df_top_10_clients_by_credit.groupby(by='customer_id')['balance'].mean().sort_values(ascending=False).head(10))

fig.update_traces(textposition='outside')
fig.update_layout(title=dict(text = "<b>Top Valuable Customers by Average Balance</b>",x=0.5,y=0.95),
                  xaxis_title="Customer ID", yaxis_title="Average Balance (Australian Dollar)",
                    title_font_size=20, showlegend=False, height = 500)

fig.update_traces(marker_coloraxis=None)
fig.show()

# 4.18  Average transaction amount by month

In [296]:
# Average transaction amount by month

df_cleaned.groupby(['hour','month','gender'])['amount'].mean()

pd.pivot_table(
    df_cleaned,
    values='amount',
    index='hour',
    columns='gender',
    aggfunc='mean'
)

df4_grp = (
    df_cleaned
    .groupby(['hour', 'month', 'gender'])
    .agg(
        Transaction_Count=('amount', 'count'),
        Total_Txn_Amount=('amount', 'sum'),
        Average_Txn_Amount=('amount', 'mean')
    )
    .reset_index()
)

# -------------------
month_gender = (
    df_cleaned
    .groupby(['month', 'gender'], as_index=False)
    .agg(
        Transaction_Count=('amount', 'count'),
        Total_Txn_Amount=('amount', 'sum'),
        Average_Txn_Amount=('amount', 'mean')
    )
)

fig = px.bar(
    month_gender,
    x='month', y='Average_Txn_Amount', barmode='group',)

fig.update_traces(textposition='outside', marker_coloraxis=None)
fig.update_xaxes(title='Month')
fig.update_yaxes(title='Average Transaction Amount in Australian Dollar')
fig.update_layout(title=dict(text = "<b>Average Transaction Amount (Australian Dollar) by Month</b>", x=0.5, y=0.95),
                    title_font_size=20, showlegend=False, height = 450,)
fig.show()

In [297]:
# Insights:

# There is a 7% increase in Avg transaction amount from August to October.
# 71% increase in AVG balance maintained by the customers.
# 77% increase in total balance over these 3 months.

In [ ]:
## 5. 